# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [17]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [ ]:
# Q1
cont = (df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
               .sort_values('Year'))
cont['Index'] = cont['Population'] / cont.groupby('Continent')['Population'].transform('first') * 100

final = cont[cont['Year'] == 2022].set_index('Continent')['Index'].sort_values(ascending=False)
leader = final.index[0]

colors = {c: '#BFBFBF' for c in final.index}
colors[leader] = '#E69F00'                     

fig = px.line(cont, x='Year', y='Index', color='Continent',
              category_orders={'Continent': list(final.index)},
              color_discrete_map=colors, markers=True,
              template='plotly_white',
              title=f'<b>{leader} has grown fastest: {final[leader] / 100:.1f}× its 1970 population</b>'
                    f'<br><sup>Population indexed to 1970 = 100 · all other continents grew 1.1–2.3×</sup>')

fig.update_traces(line=dict(width=2), marker=dict(size=5),
                  hovertemplate='%{x}: %{y:.0f}<extra>%{fullData.name}</extra>')
fig.update_traces(selector=dict(name=leader), line=dict(width=4), marker=dict(size=7))

prev = None
for c, y in final.items():
    y_lab = y if prev is None or prev - y >= 10 else prev - 10
    prev = y_lab
    fig.add_annotation(x=2022, y=y_lab,
                       text=f'<b>{c}</b> {y / 100:.1f}×' if c == leader else f'{c} {y / 100:.1f}×',
                       showarrow=False, xanchor='left', xshift=8,
                       font=dict(color=colors[c] if c == leader else '#7F7F7F',
                                 size=13 if c == leader else 12))

fig.add_hline(y=100, line_dash='dot', line_color='#7F7F7F',
              annotation_text='1970 level', annotation_position='bottom right',
              annotation_font_color='#7F7F7F')

fig.update_layout(showlegend=False, width=900, height=550,
                  margin=dict(l=60, r=150, t=90, b=40), hovermode='x unified')
fig.update_xaxes(title=None, showgrid=False,
                 tickvals=[1970, 1980, 1990, 2000, 2010, 2022], range=[1968, 2024])
fig.update_yaxes(title='Population index (1970 = 100)', showgrid=True, gridcolor='#EEEEEE',
                 zeroline=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [19]:
# Q2
top15 = (df.assign(Added=(df['2022 Population'] - df['1970 Population']) / 1e6)
           .nlargest(15, 'Added')
           .assign(Region=lambda d: np.where(d['Continent'] == 'Africa', 'Africa', 'Rest of the world')))

n_africa = (top15['Region'] == 'Africa').sum()

fig = px.bar(top15, x='Added', y='Country', orientation='h',
             color='Region',
             color_discrete_map={'Africa': '#E69F00', 'Rest of the world': '#BFBFBF'},
             text=top15['Added'].map(lambda v: f'+{v:,.0f}M'),
             template='plotly_white',
             title=f'<b>{n_africa} of the 15 countries that added the most people since 1970 are in '
                   f'<span style="color:#E69F00">Africa</span></b>'
                   f'<br><sup>Population added between 1970 and 2022, millions</sup>')

fig.update_traces(textposition='outside', cliponaxis=False,
                  hovertemplate='%{y}: +%{x:,.1f}M<extra></extra>')
fig.update_yaxes(title=None, categoryorder='total ascending', ticksuffix='  ')
fig.update_xaxes(visible=False)
fig.update_layout(showlegend=False, width=900, height=600,
                  margin=dict(l=20, r=60, t=90, b=20), bargap=0.25)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [20]:
# Q3
d = df[['Country', 'Continent', 'Area', '2022 Population']].copy()
logA, logP = np.log10(d['Area']), np.log10(d['2022 Population'])

r = np.corrcoef(logA, logP)[0, 1]
slope, intercept = np.polyfit(logA, logP, 1)
d['Residual'] = logP - (slope * logA + intercept)     

mult = 10 ** d.groupby('Continent')['Residual'].median()

outliers = pd.concat([d.nlargest(4, 'Residual'), d.nsmallest(4, 'Residual')])

palette = {'Asia': '#E69F00', 'Africa': '#0072B2', 'Europe': '#56B4E9',
           'North America': '#CC79A7', 'South America': '#000000', 'Oceania': '#999999'}

fig = px.scatter(d, x='Area', y='2022 Population', color='Continent',
                 log_x=True, log_y=True, hover_name='Country',
                 color_discrete_map=palette,
                 category_orders={'Continent': list(palette)},
                 template='plotly_white',
                 title=f'<b>Bigger countries have more people (r = {r:.2f}), '
                       f'and Asian countries sit above the trend</b>'
                       f'<br><sup>Log scales · dashed line = typical population for that area · '
                       f'Asia ≈ {mult["Asia"]:.1f}× the trend, '
                       f'Oceania & South America ≈ half</sup>')

fig.update_traces(marker=dict(size=8, opacity=0.75, line=dict(width=0.5, color='white')),
                  hovertemplate='%{hovertext}<br>Area: %{x:,.0f} km²<br>Population: %{y:,.0f}<extra></extra>')

xs = np.logspace(logA.min(), logA.max(), 50)
fig.add_scatter(x=xs, y=10 ** (slope * np.log10(xs) + intercept), mode='lines',
                line=dict(color='#555555', dash='dash', width=1.5),
                hoverinfo='skip', showlegend=False)

for _, row in outliers.iterrows():
    above = row['Residual'] > 0
    fig.add_annotation(x=np.log10(row['Area']), y=np.log10(row['2022 Population']),
                       text=row['Country'], showarrow=True, arrowhead=0, arrowwidth=1,
                       arrowcolor='#7F7F7F',
                       ax=35 if row['Country'] == 'Hong Kong' else 0,   # keep clear of Singapore
                       ay=-25 if above else 25,
                       font=dict(size=11, color='#333333'))

fig.update_xaxes(title='Land area (km², log scale)', showgrid=False)
fig.update_yaxes(title='Population 2022 (log scale)', showgrid=True, gridcolor='#EEEEEE')
fig.update_layout(width=900, height=620, legend_title_text=None,
                  legend=dict(orientation='h', y=1.0, x=0, yanchor='bottom'),
                  margin=dict(l=60, r=30, t=120, b=50))
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [21]:
# Q4
medians = df.groupby('Continent')['Density'].median().sort_values()
order = list(medians.index)   
sparsest = order[0]

colors = {c: '#9E9E9E' for c in order}
colors[sparsest] = '#0072B2' 

fig = px.box(df, x='Density', y='Continent', color='Continent',
             points='all', hover_name='Country', log_x=True,
             category_orders={'Continent': order[::-1]},
             color_discrete_map=colors,
             template='plotly_white',
             title=f'<b>{sparsest} is the least crowded continent; '
                   f'Asian & European city-states are the extremes</b>'
                   f'<br><sup>Population density per country/territory, people per km² '
                   f'(log scale) · each dot is one country</sup>')

fig.update_traces(boxmean=False, jitter=0.4, pointpos=0,
                  marker=dict(size=5, opacity=0.55), line=dict(width=1.5),
                  hovertemplate='%{hovertext}: %{x:,.1f} people/km²<extra></extra>')


for c, m in medians.items():
    fig.add_annotation(x=1.01, xref='paper', y=c,
                       text=f'median <b>{m:,.0f}</b>', showarrow=False, xanchor='left',
                       font=dict(size=11, color=colors[c] if c == sparsest else '#555555'))


for _, row in pd.concat([df.nlargest(2, 'Density'), df.nsmallest(1, 'Density')]).iterrows():
    fig.add_annotation(x=np.log10(row['Density']), y=row['Continent'], text=row['Country'],
                       showarrow=True, arrowhead=0, arrowwidth=1, arrowcolor='#7F7F7F',
                       ax=0, ay=-22, font=dict(size=10, color='#333333'))

fig.update_xaxes(title='People per km² (log scale)', showgrid=True, gridcolor='#EEEEEE',
                 tickvals=[0.01, 0.1, 1, 10, 100, 1000, 10000],
                 ticktext=['0.01', '0.1', '1', '10', '100', '1k', '10k'], range=[-2.1, 4.6])
fig.update_yaxes(title=None, showgrid=False, ticksuffix='  ')
fig.update_layout(showlegend=False, width=900, height=560,
                  margin=dict(l=20, r=110, t=90, b=50))
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [22]:
# Q5
chg = df[['CCA3', 'Country', 'Continent', '2000 Population', '2022 Population']].copy()
chg['Change (%)'] = (chg['2022 Population'] / chg['2000 Population'] - 1) * 100

bins   = [-np.inf, -10, 0, 25, 50, 100, np.inf]
labels = ['Shrank > 10%', 'Shrank 0–10%', 'Grew 0–25%', 'Grew 25–50%', 'Grew 50–100%', 'Grew > 100%']
chg['Band'] = pd.cut(chg['Change (%)'], bins=bins, labels=labels, right=False).astype(str)

band_colors = dict(zip(labels, ['#B35806', '#F1A340',
                                '#C6DBEF', '#6BAED6', '#2171B5', '#08306B']))

n_shrank = (chg['Change (%)'] < 0).sum()
n_shrank_eu = ((chg['Change (%)'] < 0) & (chg['Continent'] == 'Europe')).sum()

fig = px.choropleth(chg, locations='CCA3', locationmode='ISO-3',
                    color='Band', category_orders={'Band': labels},
                    color_discrete_map=band_colors,
                    hover_name='Country',
                    hover_data={'CCA3': False, 'Band': False, 'Change (%)': ':+.1f'},
                    template='plotly_white',
                    title=f'<b>Almost every country grew since 2000 — the {n_shrank} that shrank '
                          f'are mostly in <span style="color:#B35806">Europe</span> ({n_shrank_eu})</b>'
                          f'<br><sup>Population change 2000 → 2022 (%) · '
                          f'oranges = shrank, blues = grew, darker = larger change</sup>')

fig.update_traces(marker_line_color='white', marker_line_width=0.4)

for lon, lat, txt, col in [(-22, 50, '<b>Eastern Europe</b><br>Latvia, Lithuania,<br>Moldova ≈ −23%', '#B35806'),
                           (-22, -8, '<b>Africa</b><br>no country shrank;<br>43 of 57 grew > 50%', '#08306B'),
                           (70, 2, '<b>Gulf</b><br>Qatar +317%<br>UAE +188%', '#08306B')]:
    fig.add_scattergeo(lon=[lon], lat=[lat], text=[txt], mode='text', showlegend=False,
                       textfont=dict(size=11, color=col), hoverinfo='skip')

fig.update_geos(showframe=False, showcoastlines=False, showcountries=False,
                showland=True, landcolor='#EEEEEE', projection_type='natural earth',
                lataxis_range=[-58, 85])
fig.update_layout(width=1000, height=600, margin=dict(l=0, r=0, t=90, b=0),
                  legend=dict(title=None, orientation='h', y=-0.02, x=0.5, xanchor='center'))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [23]:
# Q6
# Q6
world = df['2022 Population'].sum()
cont_share = df.groupby('Continent')['2022 Population'].sum().div(world).mul(100).sort_values(ascending=False)
ind_chn = df.loc[df['Country'].isin(['India', 'China']), '2022 Population'].sum() / world * 100

tm = df.assign(World='World',
               Region=df['Continent'].map(lambda c: f'{c} · {cont_share[c]:.0f}%'
                                          if cont_share[c] >= 1 else f'{c} · {cont_share[c]:.1f}%'))

palette = {'Asia': '#B86E00', 'Africa': '#0072B2', 'Europe': '#6A51A3',
           'North America': '#AA4499', 'South America': '#555555', 'Oceania': '#888888'}

fig = px.treemap(tm,
                 path=['World', 'Region', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map={**palette, '(?)': '#FFFFFF'},
                 template='plotly_white',
                 title=f'<b>Asia is home to {cont_share["Asia"]:.0f}% of the world\'s '
                       f'{world/1e9:.1f} billion people — India and China alone hold '
                       f'{ind_chn:.0f}%</b>'
                       f'<br><sup>2022 population · box area = people · '
                       f'click a continent to zoom in</sup>')

fig.update_traces(root_color='#FFFFFF',
                  texttemplate='<b>%{label}</b><br>%{percentRoot:.1%}',
                  hovertemplate='<b>%{label}</b><br>%{value:,.0f} people'
                                '<br>%{percentRoot:.1%} of world<br>%{percentParent:.1%} of %{parent}'
                                '<extra></extra>',
                  marker=dict(line=dict(color='white', width=1)),
                  tiling=dict(pad=2), pathbar_visible=False,
                  insidetextfont=dict(color='white'))
fig.update_layout(width=1000, height=650, margin=dict(l=10, r=10, t=80, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [24]:
# Q7
keep = ['Africa', 'Asia', 'Europe']
shares = (df_long.assign(Group=df_long['Continent'].where(df_long['Continent'].isin(keep),
                                                          'Rest of the world'))
                 .groupby(['Year', 'Group'], as_index=False)['Population'].sum())
shares['Share (%)'] = shares['Population'] / shares.groupby('Year')['Population'].transform('sum') * 100

s = shares.pivot(index='Year', columns='Group', values='Share (%)')

order = ['Africa', 'Asia', 'Rest of the world', 'Europe']          
colors = {'Africa': '#E69F00', 'Europe': '#0072B2',               
          'Asia': '#BDBDBD', 'Rest of the world': '#E0E0E0'}       

fig = px.bar(shares, x='Year', y='Share (%)', color='Group',
             category_orders={'Group': order},
             color_discrete_map=colors,
             text=shares['Share (%)'].map(lambda v: f'{v:.0f}%'),
             template='plotly_white',
             title=f'<b><span style="color:#E69F00">Africa\'s</span> share of the world nearly doubled '
                   f'({s.loc[1970, "Africa"]:.0f}% → {s.loc[2022, "Africa"]:.0f}%) while '
                   f'<span style="color:#0072B2">Europe\'s</span> nearly halved '
                   f'({s.loc[1970, "Europe"]:.0f}% → {s.loc[2022, "Europe"]:.0f}%)</b>'
                   f'<br><sup>Share of world population by region, 1970–2022 · '
                   f'Asia stayed close to {s["Asia"].mean():.0f}% throughout</sup>')

fig.update_traces(textposition='inside', insidetextanchor='middle', textangle=0,
                  textfont=dict(size=12),
                  hovertemplate='%{x} · %{fullData.name}: %{y:.1f}%<extra></extra>')
fig.update_traces(selector=dict(name='Africa'), textfont=dict(color='black'))
fig.update_traces(selector=dict(name='Europe'), textfont=dict(color='white'))

cum = s.loc[2022, order].cumsum()
for g in order:
    fig.add_annotation(x=1.01, xref='paper', y=cum[g] - s.loc[2022, g] / 2, text=f'<b>{g}</b>',
                       showarrow=False, xanchor='left',
                       font=dict(size=12, color=colors[g] if g in ('Africa', 'Europe') else '#666666'))

fig.update_xaxes(title=None, type='category', showgrid=False)
fig.update_yaxes(visible=False, range=[0, 100])
fig.update_layout(barmode='stack', bargap=0.25, showlegend=False,
                  width=950, height=560, margin=dict(l=20, r=140, t=90, b=40))
fig.show()